# 美国财政部实际收益率来源预检

冻结来源采集、NAS回读及Qlib日历覆盖；不读取SOXX收益，不拟合，不运行账户。

In [1]:
import csv,hashlib,json,os,subprocess,tempfile
from datetime import datetime,timezone,date
from pathlib import Path
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R
from src.data.nas_market_data import publish_nas_snapshot,materialize_nas_snapshot,archive_nas_recorder,materialize_nas_evidence

root=Path.cwd();card=root/'research/stage1_treasury_real_yield_source_20261023'
plan=json.loads((card/'plan.json').read_text());assert plan['frozen'] and not (card/'run_state.json').exists()
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
raw=root/'data/snapshots/stage1_treasury_real_yield_source_20261023'
raw.mkdir(parents=True,exist_ok=True);assert not any(raw.iterdir())
state={'status':'started','plan_sha256':sha(card/'plan.json'),'external_GETs_attempted':0,'SOXX_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0}
(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];source_rows=[];failure=None
for spec in plan['exact_requests']:
    state['external_GETs_attempted']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    out=raw/spec['name'];headers=raw/(spec['name']+'.headers')
    started=datetime.now(timezone.utc).isoformat()
    command=['curl','--http1.1','--noproxy','*','--connect-timeout','10','--max-time',str(plan['request_policy']['timeout_seconds_each']),'--max-filesize',str(plan['request_policy']['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(out),'-A',plan['request_policy']['user_agent'],'-w','%{http_code} %{url_effective}',spec['url']]
    try:
        response=subprocess.run(command,capture_output=True,text=True,timeout=50)
        fields=response.stdout.split(' ',1);status=int(fields[0]) if fields and fields[0].isdigit() else None
        receipt={'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'curl_exit_code':response.returncode,'http_status':status,'final_url':fields[1] if len(fields)>1 else None,'transport_error':response.stderr[:250]}
    except Exception as exc:
        receipt={'name':spec['name'],'url':spec['url'],'requested_at_utc':started,'received_at_utc':datetime.now(timezone.utc).isoformat(),'transport_exception':type(exc).__name__+':'+str(exc)[:200]}
    receipt.update(raw_sha256=sha(out) if out.exists() else None,raw_bytes=out.stat().st_size if out.exists() else 0,headers_sha256=sha(headers) if headers.exists() else None,headers_bytes=headers.stat().st_size if headers.exists() else 0)
    receipts.append(receipt)
    if receipt.get('curl_exit_code')!=0 or receipt.get('http_status')!=200 or not out.exists() or out.stat().st_size>plan['request_policy']['max_response_bytes_each']:
        failure='transport_or_size:'+spec['name'];break
    try:
        with out.open(newline='',encoding='utf-8-sig') as handle:
            reader=csv.DictReader(handle);columns=reader.fieldnames;rows=list(reader)
        assert columns and rows
        date_col=next((x for x in columns if x.strip().lower()=='date'),None)
        yield_col=next((x for x in columns if x.strip().lower()=='10 yr'),None)
        assert date_col and yield_col,(columns[:10])
        parsed=[]
        for row in rows:
            day=datetime.strptime(row[date_col].strip(),'%m/%d/%Y').date()
            value=float(row[yield_col])
            assert -10<value<20
            parsed.append((day,value))
        days=[d for d,_ in parsed];assert len(days)==len(set(days))
        if spec['name'].startswith('par-real'):
            assert min(days).year==2003 and max(days).year==2023
        else:
            year=int(spec['name'].split('-')[-1].split('.')[0]);assert {d.year for d in days}=={year}
        assert all(pd.notna(v) for _,v in parsed)
        receipt.update(rows=len(parsed),first_date=str(min(days)),last_date=str(max(days)),date_column=date_col,yield_column=yield_col)
        source_rows.extend(parsed)
    except Exception as exc:
        failure='schema_or_year:'+spec['name']+':'+type(exc).__name__+':'+str(exc)[:160];break
    (raw/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
assert state['external_GETs_attempted']<=plan['budget']['external_GETs_max']
manifest={'schema_version':1,'experiment':plan['experiment'],'plan_sha256':state['plan_sha256'],'capture_utc':datetime.now(timezone.utc).isoformat(),'request_receipts':receipts,'failure':failure,'source_is_retrospective':True,'historical_first_public_qualified':False}
(raw/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
(card/'source_receipt.json').write_text(json.dumps(manifest,indent=2)+'\n')
files={str(p.relative_to(root)):'raw' for p in raw.iterdir() if p.is_file()}
try:
    binding=publish_nas_snapshot(workspace_root=root,source_manifest=raw/'source_manifest.json',files=files,as_of=date.today().isoformat(),source_metadata={'experiment':plan['experiment'],'source':'U.S. Treasury daily real yield current archive'})
    (card/'nas_binding.json').write_text(json.dumps(binding,indent=2)+'\n')
    with tempfile.TemporaryDirectory(prefix='treasury-cold-') as temp:
        readback=materialize_nas_snapshot(binding,workspace_root=Path(temp))
        for obj in binding['objects']:
            assert sha(Path(temp)/obj['local_path'])==obj['sha256']
    assert binding['verified'] and readback['restored_objects']==len(files)
except Exception as exc:
    failure=(failure+';' if failure else '')+'NAS:'+type(exc).__name__+':'+str(exc)[:180]
    binding=None
if failure:
    state.update(status='stopped',failure=failure)
    (card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
    result={'decision':'needs_source_or_NAS','failure':failure,'GETs':state['external_GETs_attempted'],'source_rows_received':len(source_rows),'historical_first_public_qualified':False,'SOXX_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0}
else:
    days=[d for d,_ in source_rows]
    assert len(days)==len(set(days))
    rates=pd.Series({pd.Timestamp(d):v for d,v in source_rows}).sort_index()
    provider=root/'data/qlib/full_path_tiered_allocation_20261013';os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
    qlib.init(provider_uri=str(provider),region='us',exp_manager={'class':'MLflowExpManager','module_path':'qlib.workflow.expm','kwargs':{'uri':str(card/'qlib_recorder'),'default_exp_name':plan['experiment']}})
    cal=pd.DatetimeIndex(D.calendar(start_time='2018-10-15',end_time='2026-10-09',freq='day'))
    values=[];age=[];dated=[]
    for i,day in enumerate(cal):
        bound=cal[i-2] if i>=2 else pd.NaT
        index=rates.index.searchsorted(bound,side='right')-1 if pd.notna(bound) else -1
        values.append(float(rates.iloc[index]) if index>=0 else float('nan'))
        dated.append(str(rates.index[index].date()) if index>=0 else None)
        age.append(int(i-cal.searchsorted(rates.index[index],side='left')) if index>=0 else None)
    mapped=pd.DataFrame({'qlib_session':cal,'source_date':dated,'ten_year_real_yield_percent_lag2':values,'source_age_qlib_sessions':age})
    train=mapped[mapped.qlib_session.dt.year<=2022];held=mapped[mapped.qlib_session.dt.year>=2023]
    coverage={'source_rows':len(rates),'source_first_date':str(rates.index.min().date()),'source_last_date':str(rates.index.max().date()),'qlib_sessions':len(cal),'train_sessions':len(train),'train_known':int(train.ten_year_real_yield_percent_lag2.notna().sum()),'held_sessions':len(held),'held_known':int(held.ten_year_real_yield_percent_lag2.notna().sum()),'max_source_age_qlib_sessions':int(mapped.source_age_qlib_sessions.dropna().max()),'historical_first_public_qualified':False}
    with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
        rec=R.get_recorder();rec.save_objects(artifact_path='inputs',source_manifest=manifest);rec.save_objects(artifact_path='analysis',coverage=coverage,source_map=mapped);rec.log_metrics(train_known=coverage['train_known'],held_known=coverage['held_known'],new_model_fits=0,new_native_accounts=0);rid=rec.id;eid=str(rec.experiment_id)
    evidence=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
    with tempfile.TemporaryDirectory(prefix='treasury-recorder-cold-') as temp:
        materialize_nas_evidence(card/'nas_binding.json',card=Path(temp))
        relative=Path('qlib_recorder')/eid/rid/'artifacts/analysis/source_map'
        assert sha(card/relative)==sha(Path(temp)/relative)
    result={'decision':'official_real_yield_current_history_coverage_passed_PIT_unqualified','failure':None,'GETs':state['external_GETs_attempted'],'coverage':coverage,'recorder_id':rid,'experiment_id':eid,'NAS_recorder_sha256':evidence['sha256'],'source_NAS_id':binding['source_id'],'SOXX_outcome_reads':0,'new_model_fits':0,'new_native_accounts':0,'qualified_forward_days':0}
    state.update(status='completed',recorder_id=rid)
    (card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
print(json.dumps(result))


{"decision": "official_real_yield_current_history_coverage_passed_PIT_unqualified", "failure": null, "GETs": 4, "coverage": {"source_rows": 5956, "source_first_date": "2003-01-02", "source_last_date": "2026-10-22", "qlib_sessions": 2007, "train_sessions": 1061, "train_known": 1059, "held_sessions": 946, "held_known": 946, "max_source_age_qlib_sessions": 3, "historical_first_public_qualified": false}, "recorder_id": "4d4fefd49efa4a67a3da0db8d8d378b6", "experiment_id": "512393048746701702", "NAS_recorder_sha256": "e3b506c1085dd377ff0e1b2e0c3d273dec9eee9926e4980090327ad8cb35f03e", "source_NAS_id": "e4c4b5a864d30dc8c1509aa533f95aece0ed1f327a969ad072c94d68bf3603a3", "SOXX_outcome_reads": 0, "new_model_fits": 0, "new_native_accounts": 0, "qualified_forward_days": 0}


# 美国财政部 10 年期实际收益率：Stage1 来源预检

## 中文结论

按冻结计划一次读取美国财政部四份官方 CSV，覆盖 2003-01-02 至 2026-10-22，共 5,956 个唯一来源日期。原始响应、头部、接收时间和 SHA-256 已存入 Yanxuan NAS；远端校验及定向冷恢复通过。财政部说明该利率来自收盘指示性国债报价，因此同日 09:46 开盘决策不能使用当日数值。[官方历史档案](https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rate-archives)与[口径说明](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/interest-rates-frequently-asked-questions)是来源依据。

只用冻结 Stage1 配套的原生 Qlib 交易日历，在决策日之前至少隔两个 Qlib 会话取最近的来源日期。修正后的来源映射在 2018–2022 原训练期 **1060/1061** 日有数，2023–2026 复用评价期 **944/944** 日有数。唯一未知训练日是 **2018-10-15**：配套 provider 最早只到 2018-10-12，不能在同一 Qlib 时钟下证明已有两个前序会话。首次来源映射因裁掉了日历前缀误报 1059/1061；首次修复又错误要求 1061/1061，在新 Recorder 写入前停下。两次过程均保留，第二次机械修复才得到上述保守覆盖。

这条来源可能补充 Stage1 原有 39 项价格、VIX 和广度输入，但**历史首次公开和修订版本尚未核证**。2003–2023 合并 CSV 的原始 HTTP 头显示 `Last-Modified: Wed, 14 Jan 2026 23:01:12 GMT`，可知今天收到的整份文件不是 2018–2022 逐日首次发布的原字节；此头本身不能证明某个数值被修订。两会话延迟只是保守研究假设，不能把回溯 CSV 自动变成逐日当时版本。见 `vintage_gate.json`。尚未定义收益率冲击坐标或做与旧输入的冗余检验，更未读取 SOXX 结果、拟合模型或运行 0bp 账户。下一步应先核对原始版本和决策时钟，再冻结单一坐标、接收层与信息门；只有合格候选才从完整 Stage1 f696+b472 链进入 0bp 加 Tiered 手续费的完整资金账户及 SOXX 硬门。

来源映射 Qlib Recorder `3139aaaaad954c37a938b9f23677e41a` 与 NAS 归档 SHA-256 `cb3fbbd6ee5f04273d6164efb0f8970a9a6687bfd8a64827740fe78e0bb0bf94` 已定向恢复。原始首次 Recorder `4d4fefd49efa4a67a3da0db8d8d378b6` 和错误映射仍保留。SOXX 超额、2/5bp 和独立前瞻证据均未通过；Goal 继续。
